# 01 — Explore Steam API data sources

Goal: find free (preferably keyless) endpoints for three tasks:

1. **Top ~100 games** (appid list)
2. **Current concurrent players** per game
3. **Game metadata**: name, genres, price, free-to-play, release date

Every call below hits the real API. Calls are spaced out with `time.sleep` to stay polite.
Findings are summarised in [`docs/data_sources.md`](../docs/data_sources.md).

In [1]:
import json
import time
from datetime import UTC, datetime
from zoneinfo import ZoneInfo

import httpx
import pandas as pd

BKK = ZoneInfo("Asia/Bangkok")
SLEEP_SECONDS = 1.5  # pause between calls to respect rate limits

client = httpx.Client(timeout=15, follow_redirects=True)
RATE_HEADERS = ("rate", "limit", "retry", "cache-control", "content-type", "x-eresult")


def probe(url: str, params: dict | None = None, preview: int = 400) -> httpx.Response:
    """GET a URL, print status/latency/relevant headers and a short body preview."""
    start = time.perf_counter()
    resp = client.get(url, params=params)
    elapsed = time.perf_counter() - start
    headers = {
        k: v
        for k, v in resp.headers.items()
        if any(s in k.lower() for s in RATE_HEADERS)
    }
    print(f"{resp.status_code} in {elapsed:.2f}s  {headers}")
    print(resp.text[:preview])
    time.sleep(SLEEP_SECONDS)
    return resp


print("Run started:", datetime.now(UTC).isoformat(timespec="seconds"))

Run started: 2026-11-10T09:22:19+00:00


## A. Top games

### A1. `ISteamChartsService/GetGamesByConcurrentPlayers/v1` (official Steam Web API, no key)

In [2]:
CHARTS_URL = (
    "https://api.steampowered.com/ISteamChartsService/GetGamesByConcurrentPlayers/v1/"
)
resp = probe(CHARTS_URL)
charts = resp.json()["response"]
ranks = pd.DataFrame(charts["ranks"])

last_update = datetime.fromtimestamp(charts["last_update"], UTC)
print("\nlast_update (UTC):", last_update, "| Bangkok:", last_update.astimezone(BKK))
print("games returned:", len(ranks), "| columns:", list(ranks.columns))
ranks.head(10)

200 in 0.51s  {'content-type': 'application/json; charset=UTF-8', 'x-eresult': '1', 'cache-control': 'max-age=0, no-cache, no-store'}
{"response":{"last_update":1794302297,"ranks":[{"rank":1,"appid":730,"concurrent_in_game":567044,"peak_in_game":1080352},{"rank":2,"appid":570,"concurrent_in_game":431772,"peak_in_game":707682},{"rank":3,"appid":578080,"concurrent_in_game":306717,"peak_in_game":755226},{"rank":4,"appid":4115450,"concurrent_in_game":180708,"peak_in_game":463190},{"rank":5,"appid":3419430,"concurrent_in_game":177260



last_update (UTC): 2026-11-10 09:18:17+00:00 | Bangkok: 2026-11-10 16:18:17+07:00
games returned: 100 | columns: ['rank', 'appid', 'concurrent_in_game', 'peak_in_game']


,rank,appid,concurrent_in_game,peak_in_game
0,1,730,567044,1080352
1,2,570,431772,707682
2,3,578080,306717,755226
3,4,4115450,180708,463190
4,5,3419430,177260,186948
5,6,3393110,122034,176865
6,7,4348910,100241,167948
7,8,1422450,89580,204931
8,9,1172470,85924,230678
9,10,1938090,82951,150795


In [3]:
# How fresh is the snapshot? Call again and compare last_update.
time.sleep(5)
again = client.get(CHARTS_URL).json()["response"]
print("first last_update :", charts["last_update"])
print("second last_update:", again["last_update"])
print(
    "age of snapshot (minutes):",
    round((datetime.now(UTC) - last_update).total_seconds() / 60, 1),
)

first last_update : 1794302297
second last_update: 1794302297
age of snapshot (minutes): 4.2


### A2. SteamSpy `top100in2weeks` (third-party, no key)

In [4]:
SPY_URL = "https://steamspy.com/api.php"


def steamspy(params: dict, attempts: int = 3) -> dict | None:
    """Call SteamSpy; it sometimes answers HTTP 200 with a non-JSON error body."""
    for i in range(1, attempts + 1):
        resp = client.get(SPY_URL, params=params)
        print(
            f"attempt {i}: {resp.status_code} content-type={resp.headers.get('content-type')}"
        )
        try:
            return resp.json()
        except json.JSONDecodeError:
            print("  non-JSON body:", resp.text[:120])
            time.sleep(10 * i)
    return None


spy_top = steamspy({"request": "top100in2weeks"})
if spy_top:
    spy_df = pd.DataFrame(spy_top.values())
    print("games:", len(spy_df), "| columns:", list(spy_df.columns))
    display(spy_df.head(5))

attempt 1: 200 content-type=text/html; charset=UTF-8
  non-JSON body: Connection failed: Too many connections


attempt 2: 200 content-type=text/html; charset=UTF-8
  non-JSON body: Connection failed: Too many connections


attempt 3: 200 content-type=text/html; charset=UTF-8
  non-JSON body: Connection failed: Too many connections


In [5]:
# Overlap between the two "top" lists
if spy_top:
    charts_ids = set(ranks["appid"].head(100))
    spy_ids = {int(k) for k in spy_top}
    print(
        "overlap with Steam Charts top 100:",
        len(charts_ids & spy_ids),
        "of",
        len(spy_ids),
    )
else:
    print("SteamSpy unavailable during this run - overlap not computed")

SteamSpy unavailable during this run - overlap not computed


## B. Current players — `ISteamUserStats/GetNumberOfCurrentPlayers/v1` (no key)

In [6]:
PLAYERS_URL = (
    "https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/"
)

cases = {
    "Counter-Strike 2": 730,
    "Dota 2": 570,
    "PUBG": 578080,
    "Team Fortress 2": 440,
    "Apex Legends": 1172470,
    "non-existent appid": 1,
    "a DLC (CS2 DLC)": 2678630,
}
rows = []
for label, appid in cases.items():
    resp = client.get(PLAYERS_URL, params={"appid": appid})
    rows.append(
        {
            "label": label,
            "appid": appid,
            "status": resp.status_code,
            "body": resp.text[:80],
        }
    )
    time.sleep(1)
pd.DataFrame(rows)

,label,appid,status,body
0,Counter-Strike 2,730,200,"{""response"":{""player_count"":561518,""result"":1}}"
1,Dota 2,570,200,"{""response"":{""player_count"":422447,""result"":1}}"
2,PUBG,578080,200,"{""response"":{""player_count"":306717,""result"":1}}"
3,Team Fortress 2,440,200,"{""response"":{""player_count"":78785,""result"":1}}"
4,Apex Legends,1172470,200,"{""response"":{""player_count"":85696,""result"":1}}"
5,non-existent appid,1,404,"{""response"":{""result"":42}}"
6,a DLC (CS2 DLC),2678630,200,"{""response"":{""player_count"":1,""result"":1}}"


In [7]:
# Does it agree with the Charts endpoint? (Charts snapshot may be a few minutes older.)
check = ranks.head(5)[["appid", "concurrent_in_game"]].copy()
api_counts = []
for a in check["appid"]:
    api_counts.append(
        client.get(PLAYERS_URL, params={"appid": int(a)})
        .json()["response"]
        .get("player_count")
    )
    time.sleep(1)
check["current_players_api"] = api_counts
check["diff_pct"] = (
    (check["current_players_api"] / check["concurrent_in_game"] - 1).mul(100).round(1)
)
check

,appid,concurrent_in_game,current_players_api,diff_pct
0,730,567044,561518,-1.0
1,570,431772,422447,-2.2
2,578080,306717,306717,0.0
3,4115450,180708,180708,0.0
4,3419430,177260,176634,-0.4


In [8]:
# Timing: 20 sequential calls with a 1s pause -> extrapolate to 100 games.
statuses, not_ok = [], []
start = time.perf_counter()
for appid in ranks["appid"].head(20):
    r = client.get(PLAYERS_URL, params={"appid": int(appid)})
    statuses.append(r.status_code)
    if r.status_code != 200:
        not_ok.append((int(appid), r.status_code, r.text))
    time.sleep(1)
elapsed = time.perf_counter() - start
print("status codes:", pd.Series(statuses).value_counts().to_dict())
print("non-200 (appid, status, body):", not_ok)
print(f"20 calls: {elapsed:.1f}s -> ~{elapsed * 5 / 60:.1f} min for 100 games")

status codes: {200: 19, 404: 1}
non-200 (appid, status, body): [(1422450, 404, '{"response":{"result":42}}')]
20 calls: 21.0s -> ~1.8 min for 100 games


## C. Game metadata — Store `appdetails` (no key, undocumented)

In [9]:
DETAILS_URL = "https://store.steampowered.com/api/appdetails"
resp = probe(DETAILS_URL, {"appids": 730, "cc": "us", "l": "english"}, preview=0)
data = resp.json()["730"]["data"]
print("number of top-level keys:", len(data))
print(sorted(data))

200 in 0.38s  {'content-type': 'application/json; charset=utf-8', 'cache-control': 'public,max-age=3600'}



number of top-level keys: 34
['about_the_game', 'achievements', 'background', 'background_raw', 'capsule_image', 'capsule_imagev5', 'categories', 'content_descriptors', 'detailed_description', 'developers', 'dlc', 'genres', 'header_image', 'is_free', 'linux_requirements', 'mac_requirements', 'movies', 'name', 'package_groups', 'packages', 'pc_requirements', 'platforms', 'publishers', 'ratings', 'recommendations', 'release_date', 'required_age', 'screenshots', 'short_description', 'steam_appid', 'support_info', 'supported_languages', 'type', 'website']


In [10]:
def summarise(appid: int, payload: dict) -> dict:
    """Pick the fields this project needs from one appdetails entry."""
    if not payload.get("success"):
        return {"appid": appid, "success": False}
    d = payload["data"]
    price = d.get("price_overview")
    return {
        "appid": appid,
        "success": True,
        "type": d.get("type"),
        "name": d.get("name"),
        "is_free": d.get("is_free"),
        "genres": [g["description"] for g in d.get("genres", [])],
        "price_usd": price["final"] / 100 if price else None,
        "discount_pct": price["discount_percent"] if price else None,
        "release_date": d.get("release_date"),
    }


# 730 = F2P, 1086940 = Baldur's Gate 3 (paid), 271590 = GTA V (paid), 1 = invalid
out = []
for appid in [730, 1086940, 271590, 1]:
    r = client.get(DETAILS_URL, params={"appids": appid, "cc": "us", "l": "english"})
    out.append(summarise(appid, r.json()[str(appid)]))
    time.sleep(SLEEP_SECONDS)
pd.DataFrame(out)

,appid,success,type,name,is_free,genres,price_usd,discount_pct,release_date
0,730,True,game,Counter-Strike 2,True,"[Action, Free To Play]",NaN,NaN,"{'coming_soon': False, 'date': 'Aug 21, 2012'}"
1,1086940,True,game,Baldur's Gate 3,False,"[Adventure, RPG, Strategy]",59.99,0.0,"{'coming_soon': False, 'date': 'Aug 3, 2023'}"
2,271590,True,game,Grand Theft Auto V Legacy,False,"[Action, Adventure]",NaN,NaN,"{'coming_soon': False, 'date': 'Apr 13, 2015'}"
3,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# Multiple appids in one call: without filters vs with filters=price_overview
multi = client.get(DETAILS_URL, params={"appids": "730,1086940,271590", "cc": "us"})
print("no filters    :", multi.status_code, multi.text[:150])
time.sleep(SLEEP_SECONDS)
multi_price = client.get(
    DETAILS_URL,
    params={"appids": "730,1086940,271590", "cc": "us", "filters": "price_overview"},
)
print("price_overview:", multi_price.status_code, multi_price.text[:400])
time.sleep(SLEEP_SECONDS)

no filters    : 400 null


price_overview: 200 {"730":{"success":true,"data":[]},"1086940":{"success":true,"data":{"price_overview":{"currency":"USD","initial":5999,"final":5999,"discount_percent":0,"initial_formatted":"","final_formatted":"$59.99"}}},"271590":{"success":true,"data":[]}}


In [12]:
# SteamSpy appdetails: has user tags (more granular than Store genres)
spy_app = steamspy({"request": "appdetails", "appid": 730})
if spy_app:
    print({k: spy_app.get(k) for k in ("name", "genre", "price", "ccu", "owners")})
    print("top tags:", list(spy_app.get("tags", {}))[:10])

attempt 1: 200 content-type=application/json
{'name': 'Counter-Strike: Global Offensive', 'genre': 'Action, Free To Play', 'price': '0', 'ccu': 1013936, 'owners': '100,000,000 .. 200,000,000'}
top tags: ['FPS', 'Shooter', 'Multiplayer', 'Competitive', 'Action', 'Team-Based', 'e-sports', 'Tactical', 'First-Person', 'PvP']


## D. Coverage + rate-limit check on the real top-100 list

Fetch `appdetails` for every game in the Charts top 100, 1.5 s apart (~40 calls/min,
below the community-reported ~200 calls / 5 min). This shows how many entries are
usable and whether we hit HTTP 429.

In [13]:
top100 = ranks["appid"].head(100).astype(int).tolist()
details, codes = [], []
start = time.perf_counter()
for appid in top100:
    r = client.get(DETAILS_URL, params={"appids": appid, "cc": "us", "l": "english"})
    codes.append(r.status_code)
    body = r.json() if r.status_code == 200 else None
    details.append(
        summarise(appid, body[str(appid)])
        if body
        else {"appid": appid, "success": False}
    )
    time.sleep(SLEEP_SECONDS)
elapsed = time.perf_counter() - start

det = pd.DataFrame(details)
print(
    f"{len(top100)} calls in {elapsed / 60:.1f} min | status codes: {pd.Series(codes).value_counts().to_dict()}"
)
print("success:", int(det["success"].sum()), "| failed:", int((~det["success"]).sum()))
print("type counts:", det["type"].value_counts(dropna=False).to_dict())

100 calls in 2.9 min | status codes: {200: 100}
success: 100 | failed: 0
type counts: {'game': 98, 'mod': 1, 'demo': 1}


In [14]:
ok = det[det["success"]].copy()
paid = ok[~ok["is_free"].astype(bool)]
print("free-to-play:", int(ok["is_free"].sum()), "| paid:", len(paid))
print("paid games without price_overview:", int(paid["price_usd"].isna().sum()))
print("games without genres:", int((ok["genres"].str.len() == 0).sum()))
print("coming_soon:", int(ok["release_date"].map(lambda r: r.get("coming_soon")).sum()))
print(
    "release_date samples:",
    ok["release_date"].map(lambda r: r.get("date")).head(5).tolist(),
)
print("\ngenre frequency (a game can have several genres):")
print(ok["genres"].explode().value_counts().head(12).to_string())

free-to-play: 38 | paid: 62
paid games without price_overview: 3
games without genres: 1
coming_soon: 1
release_date samples: ['Aug 21, 2012', 'Jul 9, 2013', 'Dec 21, 2017', 'Oct 28, 2026', 'Mar 5, 2025']

genre frequency (a game can have several genres):
genres
Action                   62
Adventure                42
RPG                      42
Indie                    33
Free To Play             32
Simulation               31
Strategy                 24
Massively Multiplayer    23
Casual                   20
Early Access              9
Sports                    5
Racing                    4


In [15]:
# Edge cases worth handling in the pipeline
cols = ["appid", "name", "type", "is_free", "price_usd", "genres", "release_date"]
print("non-'game' types:")
display(ok[ok["type"] != "game"][cols])
print("paid but no price_overview:")
display(paid[paid["price_usd"].isna()][cols])
print("no genres / coming soon:")
display(
    ok[
        (ok["genres"].str.len() == 0)
        | ok["release_date"].map(lambda r: r.get("coming_soon"))
    ][cols]
)
print(
    "is_free=True but no 'Free To Play' genre:",
    int(
        (
            ok["is_free"].astype(bool)
            & ~ok["genres"].map(lambda g: "Free To Play" in g)
        ).sum()
    ),
)
print(
    "Deadlock (1422450) in appdetails:",
    ok[ok["appid"] == 1422450][["name", "type", "is_free"]].to_dict("records"),
)

non-'game' types:


,appid,name,type,is_free,price_usd,genres,release_date
71,1281930,tModLoader,mod,True,NaN,"[Action, Adventure, Indie, RPG, Free To Play]","{'coming_soon': False, 'date': 'May 16, 2020'}"
98,5166840,Character House -Desktop Companion- Demo,demo,True,NaN,"[Casual, Simulation]","{'coming_soon': False, 'date': 'Sep 15, 2026'}"


paid but no price_overview:


,appid,name,type,is_free,price_usd,genres,release_date
7,1422450,Deadlock,game,False,NaN,[Action],"{'coming_soon': True, 'date': 'To be announced'}"
9,1938090,Call of Duty®,game,False,NaN,[Action],"{'coming_soon': False, 'date': 'Oct 27, 2022'}"
23,271590,Grand Theft Auto V Legacy,game,False,NaN,"[Action, Adventure]","{'coming_soon': False, 'date': 'Apr 13, 2015'}"


no genres / coming soon:


,appid,name,type,is_free,price_usd,genres,release_date
7,1422450,Deadlock,game,False,NaN,[Action],"{'coming_soon': True, 'date': 'To be announced'}"
97,1049590,Eternal Return,game,True,NaN,[],"{'coming_soon': False, 'date': 'Jul 19, 2023'}"


is_free=True but no 'Free To Play' genre: 6
Deadlock (1422450) in appdetails: [{'name': 'Deadlock', 'type': 'game', 'is_free': False}]


In [16]:
# Which top-100 entries failed (if any)?
det[~det["success"]].merge(ranks[["appid", "rank", "concurrent_in_game"]], on="appid")

,appid,success,type,name,is_free,genres,price_usd,discount_pct,release_date,rank,concurrent_in_game


In [17]:
print("Run finished:", datetime.now(UTC).isoformat(timespec="seconds"))
client.close()

Run finished: 2026-11-10T09:27:08+00:00
